# Eksperimen Final ML Stability Research

Notebook ini menjalankan eksperimen final untuk menganalisis stabilitas estimasi performa dan konsistensi peringkat beberapa algoritma klasifikasi terhadap variasi pembagian data training dan testing.

Eksperimen menggunakan:

- 3 dataset klasifikasi
- 3 algoritma
- 100 random seed
- Train-test split 80:20
- Stratified splitting
- Accuracy
- Macro F1-score

Random seed pada proses pembagian data divariasikan dari 1 sampai 100.

Random state internal model dibuat tetap agar variasi hasil terutama berasal dari perubahan komposisi data training dan testing.

In [14]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_breast_cancer

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import accuracy_score, f1_score

In [15]:
TEST_SIZE = 0.2
SEEDS = range(1, 101)

MODEL_RANDOM_STATE = 42
K = 5
N_ESTIMATORS = 100

In [16]:
wine_columns = [
    "class",
    "alcohol",
    "malic_acid",
    "ash",
    "alcalinity_of_ash",
    "magnesium",
    "total_phenols",
    "flavanoids",
    "nonflavanoid_phenols",
    "proanthocyanins",
    "color_intensity",
    "hue",
    "od280_od315",
    "proline"
]

wine_df = pd.read_csv(
    "../../data/wine/wine.data",
    header=None,
    names=wine_columns
)

X_wine = wine_df.drop("class", axis=1)
y_wine = wine_df["class"]

In [17]:
breast_data = load_breast_cancer()

X_breast = pd.DataFrame(
    breast_data.data,
    columns=breast_data.feature_names
)

y_breast = pd.Series(
    breast_data.target,
    name="target"
)

In [18]:
raisin_df = pd.read_excel(
    "../../data/raisin/Raisin_Dataset.xlsx"
)

X_raisin = raisin_df.drop("Class", axis=1)
y_raisin = raisin_df["Class"]

In [19]:
datasets = {
    "wine": (X_wine, y_wine),
    "breast_cancer": (X_breast, y_breast),
    "raisin": (X_raisin, y_raisin)
}

In [20]:
models = {
    "decision_tree": DecisionTreeClassifier(
        random_state=MODEL_RANDOM_STATE
    ),

    "knn": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", KNeighborsClassifier(
            n_neighbors=K
        ))
    ]),

    "random_forest": RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        random_state=MODEL_RANDOM_STATE
    )
}

In [21]:
hasil_final = []

for dataset_name, (X, y) in datasets.items():

    print(f"\nDataset: {dataset_name}")
    print("=" * 50)

    for seed in SEEDS:

        X_train, X_test, y_train, y_test = train_test_split(
            X,
            y,
            test_size=TEST_SIZE,
            random_state=seed,
            stratify=y
        )

        for model_name, model in models.items():

            model.fit(X_train, y_train)

            y_pred = model.predict(X_test)

            accuracy = accuracy_score(
                y_test,
                y_pred
            )

            macro_f1 = f1_score(
                y_test,
                y_pred,
                average="macro"
            )

            hasil_final.append({
                "dataset": dataset_name,
                "model": model_name,
                "seed": seed,
                "accuracy": accuracy,
                "macro_f1": macro_f1
            })

        if seed % 10 == 0:
            print(f"Seed {seed} selesai")


Dataset: wine
Seed 10 selesai
Seed 20 selesai
Seed 30 selesai
Seed 40 selesai
Seed 50 selesai
Seed 60 selesai
Seed 70 selesai
Seed 80 selesai
Seed 90 selesai
Seed 100 selesai

Dataset: breast_cancer
Seed 10 selesai
Seed 20 selesai
Seed 30 selesai
Seed 40 selesai
Seed 50 selesai
Seed 60 selesai
Seed 70 selesai
Seed 80 selesai
Seed 90 selesai
Seed 100 selesai

Dataset: raisin
Seed 10 selesai
Seed 20 selesai
Seed 30 selesai
Seed 40 selesai
Seed 50 selesai
Seed 60 selesai
Seed 70 selesai
Seed 80 selesai
Seed 90 selesai
Seed 100 selesai


In [22]:
df_final = pd.DataFrame(hasil_final)

df_final.head()

,dataset,model,seed,accuracy,macro_f1
0,wine,decision_tree,1,0.944444,0.945741
1,wine,knn,1,0.972222,0.971781
2,wine,random_forest,1,1.000000,1.000000
3,wine,decision_tree,2,0.944444,0.948413
4,wine,knn,2,0.972222,0.974321


In [23]:
df_final.shape

(900, 5)

In [24]:
df_final.groupby(
    ["dataset", "model"]
).size()

dataset        model        
breast_cancer  decision_tree    100
               knn              100
               random_forest    100
raisin         decision_tree    100
               knn              100
               random_forest    100
wine           decision_tree    100
               knn              100
               random_forest    100
dtype: int64

In [25]:
df_final.isnull().sum()

dataset     0
model       0
seed        0
accuracy    0
macro_f1    0
dtype: int64

In [26]:
assert df_final.shape == (900, 5)

assert df_final.groupby(
    ["dataset", "model"]
).size().eq(100).all()

print("Validasi eksperimen berhasil.")

Validasi eksperimen berhasil.


In [27]:
df_final.to_csv(
    "../../results/final/final_experiment_results.csv",
    index=False
)